In [ ]:
import pandas as pd
from pathlib import Path
from nd2 import ND2File, imread
from operator import add
from functools import reduce
from natsort import natsorted
from tqdm.notebook import tqdm

from calmutils.imageio import save_tiff_imagej


def repeat_file_if_necessary(img_file, multi_use_dimensions):
    """
    Return a list of repeats of img_file (nd2 file path) for each position along multi_use_dimensions.
    E.g. if multi_use_dimensions = ('P', ) and we have 16 stage positions in file, will return [img_file] * 16.
    """
    n_repeats = 1
    with ND2File(img_file) as reader:
        for dim in multi_use_dimensions:
            n_repeats *= reader.sizes.get(dim, 1)
    return [img_file] * n_repeats
    

In [ ]:
in_path = '/Volumes/agl_data/AndreasMaiser/NSD/26AM06-02_2/'

# directory containing tables with selections (e.g. regionprops with bboxes)
rprop_subdirectory = 'region_properties_nuclei'
# directory containing raw nd2 files
image_subdirectory = ''

out_subdirectory = 'patches_gfp+'

# use image files multiple times for multiple tables
# this is necessary e.g. when we have multiple XY posistions in an ND2 file and have a separate table for each
image_files_multi_use_dimensions = ('P', )

# simple (min, max) filters to select a subset of table
filters = {
    'intensity_mean': (510, None),
}

# columns names to base selection on
# can be a pair (min, max) for slice selection or a single column (single element selection)
selection_map = {
    'Y': ('bbox-0', 'bbox-2'),
    'X': ('bbox-1', 'bbox-3'),
    'P': 'image_pos'
}

# which dimension indices to include in output files (will split along these dimensions if necessary)
output_filename_dimension_prefixes = {"C": "_ch", "P": "_pos"}
# which indices from rprop table to include (e.g. cell id/label)
output_filename_rpros_prefixes = {"label": "_cell"}

In [ ]:
rprop_files = natsorted((Path(in_path) / rprop_subdirectory).glob('[!.]*.csv'))

image_files = natsorted((Path(in_path) / image_subdirectory).glob('[!.]*.nd2'))
image_files = reduce(add, [repeat_file_if_necessary(f, image_files_multi_use_dimensions) for f in image_files], [])

# image_files, rprop_files

In [ ]:

out_dir = Path(in_path) / out_subdirectory
if not out_dir.exists():
    out_dir.mkdir()


for image_file, rprop_file in tqdm(zip(image_files, rprop_files), total=len(image_files)):

    # read image and rprop table
    # read file to XArray
    with ND2File(image_file) as reader:
        img = reader.to_xarray(delayed=True)
        pixel_size = list(reader.voxel_size())[::-1]
    
    rprop_df = pd.read_csv(rprop_file)
    
    # --- filter table ---
    for column, (min_, max_) in filters.items():
        if min_ is not None:
            rprop_df = rprop_df[rprop_df[column] >= min_]
        if max_ is not None:
            rprop_df = rprop_df[rprop_df[column] <= max_]
    
    
    # --- cut patches ---
    
    for _, r in rprop_df.iterrows():
    
        patch = img
        for dim, selection in selection_map.items():
        
            # skip non existent dimension only if we also don't have selection coord in table
            # NOTE / TODO: will error if only one condition is true
            if dim not in img.dims and pd.isna(r[selection]):
                continue
        
            # pick a single coordinate
            if isinstance(selection, str):
                patch = patch.isel({dim: r[selection]})
            # pick a slice
            elif isinstance(selection, (list, tuple)):
                selection_min_column, selection_max_column = selection
                selection_min, selection_max = r[selection_min_column], r[selection_max_column]
                patch = patch.isel({dim: slice(selection_min, selection_max)})
    
        # --- split and save ---
        split_dims = patch.dims & output_filename_dimension_prefixes.keys()
        filename_idx_dims = img.dims & output_filename_dimension_prefixes.keys()
    
        
        if len(split_dims) == 0:
            patch_split = [patch]
        else:
            patch_split = [p_i for _, p_i in patch.groupby(split_dims)]
        
        for patch_i in patch_split:
        
            out_filename = image_file.stem
        
            for dim in filename_idx_dims:
                idx = img.get_index(dim).get_loc(patch_i.coords[dim].item())
                out_filename += f"{output_filename_dimension_prefixes[dim]}{idx}"
            
            for col, prefix in output_filename_rpros_prefixes.items():
                out_filename += f"{prefix}{r[col]}"   
        
            out_path = out_dir / (out_filename + '.tif')
    
            # save as tiff
            axes = "".join([d for d in patch_i.dims if d not in split_dims])
            save_tiff_imagej(out_path, patch_i.values.squeeze(), axes=axes, distance_unit="micron", pixel_size=pixel_size)
